# 🧠 10 — Time Series Modeling with LSTM

**Goal**:
In this chapter, we transition from "Tabular Regression" (LightGBM) to **"Sequence Modeling" (LSTM)**.
We aim to capture the **temporal dependencies** (e.g., pollution trends over the last 7 days) to predict the next day's AQI, specifically targeting the "rapid spike" issue.

**Refactoring Note**:
The core logic for LSTM training has been refactored into a reusable module: `src.modeling.train_lstm`.
This notebook now serves as an **orchestrator** to call these functions for multiple stations.

**Key Concepts**:
1.  **Sliding Window**: Converting time-series data into `(Samples, Time Steps, Features)` format.
2.  **LSTM (Long Short-Term Memory)**: A type of Recurrent Neural Network (RNN) designed to learn long-term dependencies.
3.  **Station-Specific Models**: Training separate models for representative stations (North, Central, South, East).

---

## ⚙️ 01 — Setup & Metadata
Import standard libraries and our custom `train_lstm` module.

---

In [ ]:
import pandas as pd
from src.config import PROCESSED_DIR
from src.features.feature_engineering import (
    clip_pollutants,
    handle_outliers_iqr,
    add_rolling_features,
    log_transform_features,
    add_time_features,
)
# Import our new LSTM module
from src.modeling.train_lstm import (
    prepare_lstm_data,
    build_lstm_model,
    train_lstm_model,
    evaluate_lstm_model,
)
from src.utils.emoji_log import success, info, task, error

## 📂 02 — Load & Preprocess Data
We load the full dataset and apply the standard feature engineering pipeline ONCE.
Note: We do NOT apply `StandardScaler` here because LSTM requires `MinMaxScaler` (0-1) which is handled inside `prepare_lstm_data`.

---

In [ ]:
# 1. Load Data
df = pd.read_parquet(PROCESSED_DIR / "full_data.parquet")
df["date"] = pd.to_datetime(df["date"])

# 2. Apply Feature Engineering Pipeline
task("Applying Feature Engineering Pipeline...")
df = clip_pollutants(df)
df = handle_outliers_iqr(df)
df = add_rolling_features(df)
df = log_transform_features(df)
df = add_time_features(df)

success(f"Data Prepared. Shape: {df.shape}")

## 🚀 03 — Multi-Station Training Loop
We select 4 representative stations (North, Central, South, East) and run the full LSTM pipeline for each.

**Pipeline Steps**:
1.  **Prepare Data**: Filter, Scale (0-1), Sliding Window.
2.  **Build Model**: Create LSTM architecture.
3.  **Train**: Fit model with Early Stopping.
4.  **Evaluate**: Predict, Inverse Transform, Plot Results.

---

In [ ]:
target_stations = ["Banqiao", "Taichung", "Kaohsiung", "Hualien"]

for station in target_stations:
    print(f"\n{'='*50}")
    print(f"🌍 Processing Station: {station}")
    print(f"{'='*50}")
    
    # 1. Prepare Data
    try:
        X_train, y_train, X_test, y_test, scaler = prepare_lstm_data(df, station)
    except ValueError as e:
        error(f"Skipping {station}: {e}")
        continue
        
    # 2. Build Model
    # X_train.shape = (Samples, TimeSteps, Features)
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = build_lstm_model(input_shape)
    
    # 3. Train
    # We use 20 epochs with Early Stopping
    history = train_lstm_model(model, X_train, y_train, station, epochs=20)
    
    # 4. Evaluate
    metrics = evaluate_lstm_model(model, X_test, y_test, scaler, station)